# HAWC+

## Functions

In [38]:
import os
import warnings
from pathlib import Path
from typing import Union, Tuple, Optional, Dict

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from astropy.io import fits
from astropy.wcs import WCS
from astropy.wcs.utils import proj_plane_pixel_scales
from astropy.utils.exceptions import AstropyWarning
from astropy.visualization import ZScaleInterval, ImageNormalize, AsinhStretch
from astropy.stats import sigma_clipped_stats
from scipy.ndimage import distance_transform_edt
from SF_analysis import estimate_mach_from_break_and_length

class HawcPolarizationAnalyzer:
    """
    Pipeline for analyzing dust polarization maps from SOFIA HAWC+. 
    Adapted for single-file multi-extension FITS format.
    """
    def __init__(self, data_directory: Union[str, Path]):
        self.base_dir = Path(data_directory)
        if not self.base_dir.exists():
            raise FileNotFoundError(f"Directory not accessible: {self.base_dir}")
    
    def load_hawc_data(self, file_path: Path) -> Tuple:
        """
        Loads I, Q, U, errors, mask, WCS, and attempts to extract FWHM and Pixel Scale
        from a single HAWC+ FITS file header.
        """
        if not file_path.exists():
            raise FileNotFoundError(f"File not found: {file_path}")
            
        with fits.open(file_path) as hdul:
            I      = np.squeeze(hdul[0].data)
            err_I  = np.squeeze(hdul[1].data)
            Q      = np.squeeze(hdul[2].data)
            err_Q  = np.squeeze(hdul[3].data)
            U      = np.squeeze(hdul[4].data)
            err_U  = np.squeeze(hdul[5].data)
            qual_I = np.squeeze(hdul[6].data) if len(hdul) > 6 else None
            
            header = hdul[0].header
            
            # --- ЧИТАЕМ ЗАГОЛОВОК С ФОЛБЕКОМ ---
            # Ключ 'BEAM' содержит приборный FWHM в arcsec
            hdr_fwhm = header.get('BEAM', None)
            
            # Ключ 'PIXSCAL' содержит размер пикселя в arcsec
            hdr_scale = header.get('PIXSCAL', None)
            
            with warnings.catch_warnings():
                warnings.simplefilter('ignore', AstropyWarning)
                wcs = WCS(header) if 'CTYPE1' in header else None
                if wcs is not None and wcs.pixel_n_dim > 2:
                    wcs = wcs.celestial
                    
        return I, Q, U, wcs, err_I, err_Q, err_U, qual_I, hdr_fwhm, hdr_scale
    
    def _get_robust_limits(self, I: np.ndarray) -> Tuple[float, float]:
        valid_I = I[np.isfinite(I)]
        if valid_I.size == 0: return 0.0, 1.0
            
        try:
            interval = ZScaleInterval()
            vmin, vmax = interval.get_limits(valid_I)
            if vmax <= vmin: raise ValueError("ZScale produced invalid limits.")
        except Exception:
            vmin, vmax = np.percentile(valid_I, [2, 98])
            
        if vmax <= vmin:
            vmin = np.nanmin(valid_I)
            vmax = np.nanmax(valid_I)
            if vmax <= vmin: vmax = vmin + 1.0 
                 
        return vmin, vmax
    
    @staticmethod
    def compute_evpa(Q: np.ndarray, U: np.ndarray) -> np.ndarray:
        return np.degrees(0.5 * np.arctan2(U, Q))

    @staticmethod
    def pol_vector_components(p: Union[float, np.ndarray], psi_rad: Union[float, np.ndarray]) -> Tuple[np.ndarray, np.ndarray]:
        psi_shifted = psi_rad + np.pi / 2.0
        return p * np.cos(psi_shifted), p * np.sin(psi_shifted)

    def get_physical_scales(self, wcs: WCS, distance_pc: float, shape: Tuple[int, int]) -> Dict[str, float]:
        pix_scale_deg = np.mean(proj_plane_pixel_scales(wcs))
        pix_scale_arcsec = pix_scale_deg * 3600.0
        pix_scale_pc = np.radians(pix_scale_deg) * distance_pc
        
        return {
            "distance_pc": distance_pc,
            "pix_scale_arcsec": pix_scale_arcsec,
            "phys_scale_pc": pix_scale_pc,
            "angular_size_arcmin": (shape[0] * pix_scale_deg * 60, shape[1] * pix_scale_deg * 60),
            "physical_size_pc": (shape[0] * pix_scale_pc, shape[1] * pix_scale_pc),
            "total_pixels": shape[0] * shape[1]
        }

    @staticmethod
    def compute_structure_function(Q: np.ndarray, U: np.ndarray, R_list: np.ndarray, 
                                   min_pI: Optional[float] = None, bin_width: float = 0.5,
                                   pixels_per_beam: float = 1.0) -> Dict[str, np.ndarray]: # <--- ДОБАВЛЕНО
        Q, U = np.asarray(Q, dtype=float), np.asarray(U, dtype=float)
        Ny, Nx = Q.shape
        pI = np.hypot(Q, U)
        
        if min_pI is None: min_pI = -np.inf
            
        valid = np.isfinite(Q) & np.isfinite(U) & (pI > min_pI)
        q = np.where(valid, Q / pI, np.nan)
        u = np.where(valid, U / pI, np.nan)

        valid_R, valid_Dphi, valid_Npairs, valid_SF_err = [], [], [], []

        def accumulate_shift(dy: int, dx: int) -> Optional[np.ndarray]:
            # Вычисляем индексы для оригинального массива (q1, u1) и сдвинутого (q2, u2)
            y1s, y2s = slice(max(0, dy), min(Ny, Ny + dy)), slice(max(0, -dy), min(Ny, Ny - dy))
            x1s, x2s = slice(max(0, dx), min(Nx, Nx + dx)), slice(max(0, -dx), min(Nx, Nx - dx))
            
            q1, u1 = q[y1s, x1s], u[y1s, x1s]
            q2, u2 = q[y2s, x2s], u[y2s, x2s]
            
            # [+] ПРОВЕРКА: Если срез пуст (размерность 0 по оси), возвращаем None
            if q1.size == 0 or q2.size == 0 or q1.shape != q2.shape:
                return None
                
            m = np.isfinite(q1) & np.isfinite(q2)
            if not np.any(m): return None
            
            dq, du = q1[m] - q2[m], u1[m] - u2[m]
            return 0.25 * (dq**2 + du**2)

        for R in R_list:
            if R < 0: continue
            rmax = int(np.ceil(R + bin_width))
            shifts = [(dy, dx) for dy in range(-rmax, rmax + 1) for dx in range(-rmax, rmax + 1) 
                      if (dx != 0 or dy != 0) and abs(np.hypot(dx, dy) - R) <= bin_width]

            contrib_list = [accumulate_shift(dy, dx) for dy, dx in shifts]
            valid_contribs = [c for c in contrib_list if c is not None and c.size > 0]
            
            if valid_contribs:
                contrib_vec = np.concatenate(valid_contribs)
                count = contrib_vec.size
                
                # ИСПОЛЬЗУЕМ Neff
                N_eff = count / pixels_per_beam
                
                if N_eff < 5: break
                    
                mean_val = np.nanmean(contrib_vec)
                std_dev = np.nanstd(contrib_vec, ddof=1)
                
                # Истинная стандартная ошибка с учетом корреляции пикселей
                sem = std_dev / np.sqrt(N_eff)
                relative_error = sem / (mean_val + 1e-9)
                
                #if relative_error > 0.10: break 
                
                valid_R.append(R)
                valid_Dphi.append(mean_val)
                valid_Npairs.append(count)
                valid_SF_err.append(sem)
            else:
                break

        if len(valid_R) == 0: return None

        return {
            "R_px": np.array(valid_R), 
            "SF": np.array(valid_Dphi), 
            "SF_err": np.array(valid_SF_err),
            "Npairs": np.array(valid_Npairs)
        }
    # ==========================================
    # ПЛОТТЕРЫ 
    # ==========================================
    def plot_polarization_map(self, I: np.ndarray, vx: np.ndarray, vy: np.ndarray, wcs: WCS, object_name: str, step: int = 4, save_path: str = None):
        vmin, vmax = self._get_robust_limits(I)
        x = np.linspace(0, vx.shape[1] - 1, vx.shape[1])
        y = np.linspace(0, vx.shape[0] - 1, vx.shape[0])
        px, py = np.meshgrid(x, y)

        fig = plt.figure(figsize=(7, 6))
        ax = fig.add_subplot(111, projection=wcs)
        im = ax.imshow(I, origin="lower", vmin=vmin, vmax=vmax, cmap='viridis')
        
        ax.quiver(px[::step, ::step], py[::step, ::step], vx[::step, ::step], vy[::step, ::step],
                  angles='xy', scale_units='xy', scale=0.2, headwidth=0, headlength=0, headaxislength=0,
                  pivot="mid", width=0.003, color="k")

        plt.colorbar(im, ax=ax, pad=0.02, label=r"Stokes $I$")
        ax.set_xlabel("RA (J2000)")
        ax.set_ylabel("Dec (J2000)")
        ax.set_title(rf"{object_name} -- Magnetic Field Morphology")
        if save_path:
            fig.savefig(save_path, bbox_inches='tight')
            plt.close(fig)
        else:
            pass#plt.show()
    
    def plot_snr_distributions(self, p_frac, psi_deg, snr_pI, object_name, save_path):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
        mask = np.isfinite(p_frac) & np.isfinite(psi_deg)
        
        ax1.hist(p_frac[mask], bins=30, color='royalblue', alpha=0.7, density=True)
        ax1.set_xlabel("Polarization Fraction p")
        
        ax2.hist(psi_deg[mask], bins=36, color='crimson', alpha=0.7, density=True)
        ax2.set_xlabel("EVPA (deg)")
        
        fig.suptitle(f"{object_name} SNR Distribution")
        plt.savefig(f"{save_path}.pdf")
        plt.close()
    def detect_regions_by_snr(self, I, Q, U, err_Q, err_U, qual_I, wcs, distance_pc, object_name, 
                              fwhm_arcsec=None, pix_scale_fallback=None, snr_threshold=3.0, 
                              min_beams=5.0, save_path=None):
        """
        Сегментация по SNR с динамическим определением размера луча из хедера или вручную.
        """
        scales = self.get_physical_scales(wcs, distance_pc, I.shape)
        
        # ЛОГИКА ДЕФОЛТА ДЛЯ PIXEL SCALE
        # Предпочитаем честный расчет WCS, затем ключ PIXSCAL, затем ручной ввод
        pix_scale = scales.get("pix_scale_arcsec", pix_scale_fallback)
        if pix_scale is None:
            pix_scale = 4.55  # совсем крайний случай
            print(f"[WARNING] No pixel scale found. Using ultimate default: {pix_scale} arcsec/px")

        # ЛОГИКА ДЕФОЛТА ДЛЯ FWHM
        if fwhm_arcsec is None:
            fwhm_arcsec = 14.1  # Дефолт, если в FITS нет ключа BEAM
            print(f"[WARNING] No FWHM found in header. Using default: {fwhm_arcsec} arcsec")
            
        print(f"-> Using Physics Scales: FWHM = {fwhm_arcsec}\", Pixel Scale = {pix_scale:.2f}\"/px")

        ppb = 1.133 * (fwhm_arcsec / pix_scale)**2  # Пикселей в одном луче (N_beam)
        min_area_pixels = min_beams * ppb 
           # Динамический порог площади
        
        # 2. Подготовка карт ошибок HAWC+
        err_Q_map = np.where((err_Q > 0) & np.isfinite(err_Q), err_Q, np.inf)
        err_U_map = np.where((err_U > 0) & np.isfinite(err_U), err_U, np.inf)
        if qual_I is not None:
            invalid = (qual_I <= 0) | ~np.isfinite(qual_I)
            err_Q_map[invalid] = np.inf
            err_U_map[invalid] = np.inf

        # 3. Расчет дебиасированного SNR
        pI_raw = np.hypot(Q, U)
        pI_safe = np.where(pI_raw > 0, pI_raw, np.inf)
        noise_pI = np.sqrt(Q**2 * err_Q_map**2 + U**2 * err_U_map**2) / pI_safe
        radicand = pI_raw**2 - noise_pI**2
        pI_debiased = np.sqrt(np.maximum(radicand, 0))
        snr_map = np.where(noise_pI > 0, pI_debiased / noise_pI, 0)

        # 4. Бинарная маска и морфология
        snr_mask = (snr_map >= snr_threshold).astype(np.uint8) * 255
        kernel = np.ones((3, 3), np.uint8)
        snr_mask = cv2.morphologyEx(snr_mask, cv2.MORPH_OPEN, kernel, iterations=1)

        # 5. Фильтрация контуров
        contours, _ = cv2.findContours(snr_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        valid_contours = []
        for cnt in contours:
            area = cv2.contourArea(cnt)
            if area >= min_area_pixels:
                valid_contours.append(cnt)
                
        valid_contours = sorted(valid_contours, key=cv2.contourArea, reverse=True)

        # 6. Отрисовка диагностической карты
        fig = plt.figure(figsize=(8, 7))
        ax = fig.add_subplot(111, projection=wcs)
        norm = ImageNormalize(I, interval=ZScaleInterval(), stretch=AsinhStretch(a=0.1))
        im = ax.imshow(I, origin='lower', cmap='viridis', norm=norm)

        count = len(valid_contours)
        for idx, cnt in enumerate(valid_contours):
            poly = cnt.reshape(-1, 2)
            poly = np.vstack([poly, poly[0]])
            ax.plot(poly[:, 0], poly[:, 1], color='red', lw=1.5)
            M = cv2.moments(cnt)
            cX, cY = (int(M["m10"] / M["m00"]), int(M["m01"] / M["m00"])) if M["m00"] != 0 else (int(poly[0, 0]), int(poly[0, 1]))
            ax.text(cX, cY, str(idx + 1), color='white', fontsize=11, fontweight='bold', ha='center', va='center',
                    bbox=dict(facecolor='black', alpha=0.75, edgecolor='none', pad=1.5))

        ax.set_title(f"{object_name}: {count} SNR>{snr_threshold} regions (>{min_beams} beams)")
        plt.colorbar(im, ax=ax, pad=0.02, label=r"Stokes $I$")

        if save_path:
            plt.savefig(str(save_path), bbox_inches='tight')
            plt.close(fig)
        else:
            pass#plt.show()

        return valid_contours, snr_map, err_Q_map, err_U_map, ppb
    
    def detect_regions_by_snr(self, I, Q, U, err_Q, err_U, qual_I, wcs, distance_pc, object_name, 
                              fwhm_arcsec=None, pix_scale_fallback=None, snr_threshold=3.0, 
                              min_beams=5.0, save_path=None):
        """
        Сегментация по SNR с динамическим определением порога площади (Медиана + 1.5 сигмы).
        """
        scales = self.get_physical_scales(wcs, distance_pc, I.shape)
        
        # ЛОГИКА ДЕФОЛТА ДЛЯ PIXEL SCALE И FWHM
        pix_scale = scales.get("pix_scale_arcsec", pix_scale_fallback)
        if pix_scale is None:
            pix_scale = 4.55 
            print(f"[WARNING] No pixel scale found. Using ultimate default: {pix_scale} arcsec/px")

        if fwhm_arcsec is None:
            fwhm_arcsec = 14.1 
            print(f"[WARNING] No FWHM found in header. Using default: {fwhm_arcsec} arcsec")
            
        print(f"-> Using Physics Scales: FWHM = {fwhm_arcsec}\", Pixel Scale = {pix_scale:.2f}\"/px")

        ppb = 1.133 * (fwhm_arcsec / pix_scale)**2  # Пикселей в одном луче (N_beam)
        
        # ==========================================
        # 1. РАСЧЕТ ДЕБИАСИРОВАННОГО SNR И ОШИБОК
        # ==========================================
        err_Q_map = np.where((err_Q > 0) & np.isfinite(err_Q), err_Q, np.inf)
        err_U_map = np.where((err_U > 0) & np.isfinite(err_U), err_U, np.inf)
        if qual_I is not None:
            invalid = (qual_I <= 0) | ~np.isfinite(qual_I)
            err_Q_map[invalid] = np.inf
            err_U_map[invalid] = np.inf

        pI_raw = np.hypot(Q, U)
        pI_safe = np.where(pI_raw > 0, pI_raw, np.inf)
        noise_pI = np.sqrt(Q**2 * err_Q_map**2 + U**2 * err_U_map**2) / pI_safe
        radicand = pI_raw**2 - noise_pI**2
        pI_debiased = np.sqrt(np.maximum(radicand, 0))
        snr_map = np.where(noise_pI > 0, pI_debiased / noise_pI, 0)
        print("------> The countours searching is started")
        # ==========================================
        # 2. ПОИСК ВСЕХ КОНТУРОВ
        # ==========================================
        snr_mask = (snr_map >= snr_threshold).astype(np.uint8) * 255
        kernel = np.ones((3, 3), np.uint8)
        snr_mask = cv2.morphologyEx(snr_mask, cv2.MORPH_OPEN, kernel, iterations=1)

        all_contours, _ = cv2.findContours(snr_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        # ==========================================
        # 3. ФИЗИКО-АЛГОРИТМИЧЕСКИЙ ПОРОГ ПЛОЩАДИ
        # ==========================================
        # Требование 1: Физическое (Spatial Decorrelation Limit)
        # Чтобы структурная функция зафиксировала турбулентность вне аппаратного луча, 
        # диаметр области должен быть не менее ~4.5 лучей.
        # Площадь S = pi * (D/2)^2 ≈ 16 beams.
        physical_min_beams = 16.0 
        physical_min_area = physical_min_beams * ppb

        # Требование 2: Алгоритмическое (SF min_points limit)
        # Для успешного фита SFAnalysisConfig требует min_points=8.
        # Т.к. R_max = (2/3)*L_spine, то L_spine >= 12 пикселей. Площадь ~ L^2.
        sf_min_points = 8
        algorithmic_min_area = (sf_min_points * 1.5)**2

        # Берем максимум из двух условий, чтобы удовлетворить и физику, и математику
        min_area_pixels = max(physical_min_area, algorithmic_min_area)

        print(f"-> Area Thresholds: Physical ({physical_min_beams} beams) = {physical_min_area:.1f} px, "
              f"Algorithmic = {algorithmic_min_area:.1f} px")
        print(f"-> Final Min Area Threshold set to: {min_area_pixels:.1f} pixels")


        # ==========================================
        # 4. ФИЛЬТРАЦИЯ
        # ==========================================
        valid_contours = []
        for cnt in all_contours:
            if cv2.contourArea(cnt) >= min_area_pixels:
                valid_contours.append(cnt)
                
        valid_contours = sorted(valid_contours, key=cv2.contourArea, reverse=True)

        # ==========================================
        # 5. ОТРИСОВКА
        # ==========================================
        fig = plt.figure(figsize=(8, 7))
        ax = fig.add_subplot(111, projection=wcs)
        norm = ImageNormalize(I, interval=ZScaleInterval(), stretch=AsinhStretch(a=0.1))
        im = ax.imshow(I, origin='lower', cmap='viridis', norm=norm)

        count = len(valid_contours)
        for idx, cnt in enumerate(valid_contours):
            poly = cnt.reshape(-1, 2)
            poly = np.vstack([poly, poly[0]])
            ax.plot(poly[:, 0], poly[:, 1], color='red', lw=1.5)
            M = cv2.moments(cnt)
            cX, cY = (int(M["m10"] / M["m00"]), int(M["m01"] / M["m00"])) if M["m00"] != 0 else (int(poly[0, 0]), int(poly[0, 1]))
            ax.text(cX, cY, str(idx + 1), color='white', fontsize=11, fontweight='bold', ha='center', va='center',
                    bbox=dict(facecolor='black', alpha=0.75, edgecolor='none', pad=1.5))

        ax.set_title(f"{object_name}: {count} SNR>{snr_threshold} regions (Area >= {min_area_pixels:.0f} px)")
        plt.colorbar(im, ax=ax, pad=0.02, label=r"Stokes $I$")

        if save_path:
            plt.savefig(str(save_path), bbox_inches='tight')
            plt.close(fig)
        else:
            pass

        # Окончание функции detect_regions_by_snr
        return valid_contours, snr_map, err_Q_map, err_U_map, ppb, min_area_pixels
 
    def plot_snr_colored_polarization_map(self, I: np.ndarray, vx: np.ndarray, vy: np.ndarray, 
                                          snr_map: np.ndarray, wcs: WCS, valid_contours: list, 
                                          object_name: str, step: int = 4, save_path: str = None):
        """
        Отрисовывает карту интенсивности с наложением ВСЕХ валидных контуров сегментации 
        тонкой линией цвета magenta и векторов поляризации, раскрашенных по бинам SNR.
        """
        vmin, vmax = self._get_robust_limits(I)
        x = np.arange(vx.shape[1])
        y = np.arange(vx.shape[0])
        px, py = np.meshgrid(x, y)

        fig = plt.figure(figsize=(8, 7))
        ax = fig.add_subplot(111, projection=wcs)
        
        # Фон в оттенках серого для максимального контраста цветных векторов
        norm = ImageNormalize(I, interval=ZScaleInterval(), stretch=AsinhStretch(a=0.1))
        im = ax.imshow(I, origin="lower", cmap='gray', norm=norm, alpha=0.8)

        # 1. Отрисовка всех контуров единым стилем: тонкая линия magenta
        for cnt in valid_contours:
            poly = cnt.reshape(-1, 2)
            poly = np.vstack([poly, poly[0]]) # Замыкаем полигон
            ax.plot(poly[:, 0], poly[:, 1], color='magenta', lw=.5, alpha=0.7, linestyle='solid')

        # 2. Подготовка пространственной сетки для векторов (step)
        grid_mask = np.zeros_like(snr_map, dtype=bool)
        grid_mask[::step, ::step] = True

        # 3. Фильтрация по бинам SNR
        snr_bin1 = (snr_map >= 3) & (snr_map < 5) & grid_mask
        snr_bin2 = (snr_map >= 5) & (snr_map < 10) & grid_mask
        snr_bin3 = (snr_map >= 10) & grid_mask

        # Общие параметры для векторов
        q_kwargs = dict(angles='xy', scale_units='xy', scale=0.2, headwidth=0, 
                        headlength=0, headaxislength=0, pivot="mid", width=0.003)

        # Отрисовка слоев векторов
        if np.any(snr_bin1):
            ax.quiver(px[snr_bin1], py[snr_bin1], vx[snr_bin1], vy[snr_bin1], 
                      color="cyan", label=r"$3 \leq SNR < 5$", **q_kwargs)
        if np.any(snr_bin2):
            ax.quiver(px[snr_bin2], py[snr_bin2], vx[snr_bin2], vy[snr_bin2], 
                      color="yellow", label=r"$5 \leq SNR < 10$", **q_kwargs)
        if np.any(snr_bin3):
            ax.quiver(px[snr_bin3], py[snr_bin3], vx[snr_bin3], vy[snr_bin3], 
                      color="red", label=r"$SNR \geq 10$", **q_kwargs)

        plt.colorbar(im, ax=ax, pad=0.02, label=r"Stokes $I$")
        ax.set_xlabel("RA (J2000)")
        ax.set_ylabel("Dec (J2000)")
        ax.set_title(rf"{object_name} -- SNR-Coded Polarization Vectors")
        
        # Легенда с черным фоном для читабельности
        ax.legend(loc='best', facecolor='black', labelcolor='white', framealpha=0.75)

        if save_path:
            fig.savefig(save_path, bbox_inches='tight')
            plt.close(fig)
        else:
            pass#plt.show()       
    def plot_snr_colored_polarization_map(self, I: np.ndarray, vx: np.ndarray, vy: np.ndarray, 
                                          snr_map: np.ndarray, wcs: WCS, valid_contours: list, 
                                          object_name: str, step: int = 4, save_path: str = None):
        """
        Отрисовывает карту интенсивности с наложением ВСЕХ валидных контуров сегментации 
        тонкой линией magenta (с сохранением нумерации ядер) и векторов поляризации, 
        раскрашенных по бинам SNR.
        """
        vmin, vmax = self._get_robust_limits(I)
        x = np.arange(vx.shape[1])
        y = np.arange(vx.shape[0])
        px, py = np.meshgrid(x, y)

        fig = plt.figure(figsize=(8, 7))
        ax = fig.add_subplot(111, projection=wcs)
        
        # Фон в оттенках серого для максимального контраста цветных векторов
        norm = ImageNormalize(I, interval=ZScaleInterval(), stretch=AsinhStretch(a=0.1))
        im = ax.imshow(I, origin="lower", cmap='gray', norm=norm, alpha=0.8)

        # 1. Отрисовка контуров (magenta, lw=1.0) и их нумерации
        for idx, cnt in enumerate(valid_contours):
            poly = cnt.reshape(-1, 2)
            poly = np.vstack([poly, poly[0]]) # Замыкаем полигон
            
            # Рисуем тонкий контур
            ax.plot(poly[:, 0], poly[:, 1], color='magenta', lw=1.0, alpha=0.8, linestyle='solid')
            
            # Считаем центроид для вывода номера области
            M = cv2.moments(cnt)
            cX, cY = (int(M["m10"] / M["m00"]), int(M["m01"] / M["m00"])) if M["m00"] != 0 else (int(poly[0, 0]), int(poly[0, 1]))
            
            # Небольшой контрастный маркер с номером ядра
            ax.text(cX, cY, str(idx + 1), color='white', fontsize=9, fontweight='bold', ha='left', va='top',
                    bbox=dict(facecolor='black', alpha=0.75, edgecolor='none', pad=1.0))

        # 2. Подготовка пространственной сетки для векторов (step)
        grid_mask = np.zeros_like(snr_map, dtype=bool)
        grid_mask[::step, ::step] = True

        # 3. Фильтрация по бинам SNR
        snr_bin1 = (snr_map >= 3) & (snr_map < 5) & grid_mask
        snr_bin2 = (snr_map >= 5) & (snr_map < 10) & grid_mask
        snr_bin3 = (snr_map >= 10) & grid_mask

        # Общие параметры для векторов
        q_kwargs = dict(angles='xy', scale_units='xy', scale=0.2, headwidth=0, 
                        headlength=0, headaxislength=0, pivot="mid", width=0.003)

        # Отрисовка слоев векторов
        if np.any(snr_bin1):
            ax.quiver(px[snr_bin1], py[snr_bin1], vx[snr_bin1], vy[snr_bin1], 
                      color="cyan", label=r"$3 \leq SNR < 5$", **q_kwargs)
        if np.any(snr_bin2):
            ax.quiver(px[snr_bin2], py[snr_bin2], vx[snr_bin2], vy[snr_bin2], 
                      color="yellow", label=r"$5 \leq SNR < 10$", **q_kwargs)
        if np.any(snr_bin3):
            ax.quiver(px[snr_bin3], py[snr_bin3], vx[snr_bin3], vy[snr_bin3], 
                      color="red", label=r"$SNR \geq 10$", **q_kwargs)

        plt.colorbar(im, ax=ax, pad=0.02, label=r"Stokes $I$")
        ax.set_xlabel("RA (J2000)")
        ax.set_ylabel("Dec (J2000)")
        ax.set_title(rf"{object_name} -- SNR-Coded Polarization Vectors")
        
        # Легенда с черным фоном для читабельности
        ax.legend(loc='best', facecolor='black', labelcolor='white', framealpha=0.75)

        if save_path:
            fig.savefig(save_path, bbox_inches='tight')
            plt.close(fig)
        else:
            pass#plt.show()
        

    def process_hawc_file(self, file_path: Path, distance_pc: float, 
                          fwhm_fallback: float = 14.1, pix_scale_fallback: float = 4.55,
                          snr_threshold: float = 3.0, min_beams: float = 5.0, step: int = 2):
        from SF_analysis import (
            SFAnalysisConfig, analyze_sf_table, estimate_mach_from_break_and_length,
            plot_sf_diagnostic
        )

        object_name = file_path.stem
        print(f"\n{'='*60}\nPROCESSING: {object_name}\n{'='*60}")

        try:
            # Считываем данные и забираем fwhm/scale из хедера, если они там есть
            I, Q, U, wcs, err_I, err_Q, err_U, qual_I, hdr_fwhm, hdr_scale = self.load_hawc_data(file_path)
        except Exception as e:
            print(f"[ERROR] Failed to load {file_path.name}: {e}")
            return None, None, None

        # --- ИЕРАРХИЯ ВЫБОРА: ХЕДЕР VS РУЧНОЙ ВВОД ---
        chosen_fwhm = hdr_fwhm if hdr_fwhm is not None else fwhm_fallback
        chosen_scale = hdr_scale if hdr_scale is not None else pix_scale_fallback

        report_dir = self.base_dir / "hawc_report"
        plots_dir = report_dir / "plots" / object_name
        plots_dir.mkdir(parents=True, exist_ok=True)
        scales = self.get_physical_scales(wcs, distance_pc, I.shape)

        # --- ЭТАП 1: ФИЗИЧЕСКАЯ СЕГМЕНТАЦИЯ ---
        diag_path = plots_dir / f"{object_name}_stat_segmentation.pdf"
        valid_contours, snr_pI_map, err_Q_map, err_U_map, ppb, min_area_pixels = self.detect_regions_by_snr(
            I, Q, U, err_Q, err_U, qual_I, wcs, distance_pc, object_name,
            fwhm_arcsec=chosen_fwhm, pix_scale_fallback=chosen_scale, 
            snr_threshold=snr_threshold, min_beams=min_beams, save_path=str(diag_path)
        )

        if not valid_contours:
            print(f"[ERROR] No valid SNR>{snr_threshold} clumps found. Physics canceled.")
            return None, None, None

        # --- ЭТАП 2: ИЗОЛЯЦИЯ ВСЕХ ВАЛИДНЫХ ОБЛАСТЕЙ ---
        science_mask = np.zeros(I.shape, dtype=np.uint8)
        for cnt in valid_contours:
            cv2.drawContours(science_mask, [cnt], -1, 1, thickness=cv2.FILLED)
        science_mask_bool = science_mask.astype(bool)

        I_full = I.copy()
        I[~science_mask_bool] = np.nan
        Q[~science_mask_bool] = np.nan
        U[~science_mask_bool] = np.nan

        psi_deg = self.compute_evpa(Q, U)
        vx, vy = self.pol_vector_components(p=1.0, psi_rad=np.radians(psi_deg))

        # --- ЭТАП 3: ДИНАМИЧЕСКОЕ СОЗДАНИЕ МАСОК ДЛЯ КАЖДОГО КЛАНГА ---
        # Больше никаких ручных Core_1 и Core_2. Сколько нашли - столько заносим в анализ.
        masks_to_analyze = {}
        combined_mask = np.zeros(I.shape, dtype=bool)

        for idx, cnt in enumerate(valid_contours):
            mask_clump = np.zeros(I.shape, dtype=np.uint8)
            cv2.drawContours(mask_clump, [cnt], -1, 1, thickness=cv2.FILLED)
            mask_bool = mask_clump.astype(bool)
            
            masks_to_analyze[f"Core_{idx+1}"] = mask_bool
            combined_mask |= mask_bool  # Накапливаем общую маску

        # Если областей несколько, добавляем комбинированную маску для глобального тренда
        if len(valid_contours) > 1:
            masks_to_analyze["Combined_All_Cores"] = combined_mask

        sf_results_dict = {}
        sf_config = SFAnalysisConfig(
            r_col="R_pix", dphi_col="Dphi", npairs_col="Npairs", r_pc_col="R_pc",
            min_points=8, min_npairs=30, tail_fraction=0.25, smooth_window=3,
            rel_tol=0.08, sigma_tol=2.0, require_consecutive_tail_points=3, start_offset_pix=3
        )

        # --- ЭТАП 4: СТАТИСТИЧЕСКИЙ АНАЛИЗ SF ПО ВСЕМ СОЗДАННЫМ МАСКАМ ---
        for region_name, current_mask in masks_to_analyze.items():
            y_idx, x_idx = np.where(current_mask)
            if len(y_idx) == 0: continue
                
            dx, dy = x_idx.max() - x_idx.min(), y_idx.max() - y_idx.min()
            L_spine_pix = max(dx, dy) 
            local_max_R = max(5, int(L_spine_pix / 3 * 2)) 
            R_list_px = np.arange(1, local_max_R + 1)
            
            I_masked = np.where(current_mask, I, np.nan)
            Q_masked = np.where(current_mask, Q, np.nan)
            U_masked = np.where(current_mask, U, np.nan)
                
            pI_local = np.hypot(Q_masked, U_masked)
            noise_pI_loc = np.sqrt(Q_masked**2 * err_Q_map**2 + U_masked**2 * err_U_map**2) / np.where(pI_local > 0, pI_local, np.inf)
            
            p_frac_local = pI_local / I_masked
            psi_deg_local = self.compute_evpa(Q_masked, U_masked)
            snr_pI_local = np.where(noise_pI_loc > 0, pI_local / noise_pI_loc, 0)

            self.plot_snr_distributions(
                p_frac_local, psi_deg_local, snr_pI_local,
                object_name=f"{object_name} {region_name}",
                save_path=plots_dir / f"{object_name}_snr_dist_{region_name}"
            )
            
            sf_data = self.compute_structure_function(
                Q_masked, U_masked, R_list=R_list_px, pixels_per_beam=ppb 
            )
            
            if sf_data is not None:
                sf_data["R_pc"] = R_list_px * scales["phys_scale_pc"]
                df_sf = pd.DataFrame({"R_pix": sf_data["R_px"], "Dphi": sf_data["SF"], 
                                      "Npairs": sf_data["Npairs"], "R_pc": sf_data["R_pc"]})

                # 1. Вызываем базовый анализ структурной функции
                result, valid, extras = analyze_sf_table(df_sf, sf_config, plots_dir)
                result.object_name = object_name
                result.mask_name = region_name

                # 2. Извлекаем корректную точку излома lA (saturation_scale_pix)
                # Если плато не найдено, r_break_pix будет None
                r_break_pix = result.saturation_scale_pix 

                # 3. ЧЕСТНЫЙ ФИЗИЧЕСКИЙ РАСЧЕТ ЧИСЛА МАХА
                # L — это строго L_spine_pix (макромасштаб области/облака)
                # lA — это строго r_break_pix (микромасштаб излома)
                mach_result = estimate_mach_from_break_and_length(
                    r_break_pix=r_break_pix, 
                    L_pix=L_spine_pix, 
                    object_name=object_name, 
                    mask_name=region_name
                )

                # Инициализируем значения для записи
                calculated_ma = mach_result.MA_proxy if mach_result.status == "ok" else np.nan

                # Обновляем массивы результатов
                sf_data["MA_proxy"] = calculated_ma
                sf_data["L_fit_px"] = L_spine_pix
                sf_data["lA_fit_px"] = r_break_pix
                sf_results_dict[region_name] = sf_data
                
                if len(valid) > 0:
                    # 4. Безопасное вычисление статистики интенсивности для участка
                    valid_I = I_masked[np.isfinite(I_masked)]
                    area_pix = len(valid_I)
                    i_max = float(np.nanmax(valid_I)) if area_pix > 0 else np.nan
                    i_min = float(np.nanmin(valid_I)) if area_pix > 0 else np.nan
                    i_med = float(np.nanmedian(valid_I)) if area_pix > 0 else np.nan

                    # 5. Сборка расширенного словаря для записи в CSV
                    out_dict = result.__dict__.copy()
                    out_dict.update({
                        "Distance": distance_pc,
                        "MA_proxy": calculated_ma,   # Записываем честно посчитанное число Маха
                        "L_fit_px": L_spine_pix,     # Длина продольной области
                        "lA_fit_px": r_break_pix,    # Альфвеновский масштаб перехода
                        "Area_pix": area_pix,
                        "I_max": i_max,
                        "I_min": i_min,
                        "I_median": i_med,
                        "R_max_pix": local_max_R,
                        "Min_Area_Threshold_px": min_area_pixels
                    })
    
                    # Сохранение расширенного словаря в файл
                    pd.DataFrame([out_dict]).to_csv(plots_dir / f"{object_name}_{region_name}_analysis.csv", index=False)
                    
                    plot_sf_diagnostic(valid, result, extras, sf_config, 
                                       title=f"SF: {object_name} ({region_name.replace('_', ' ')})",
                                       save_path=plots_dir / f"{object_name}_sf_{region_name}.pdf")
        # --- ЭТАП 5: ОТРИСОВКА КАРТ ---
        self.plot_polarization_map(I_full, vx, vy, wcs, f"{object_name} (Full Context)", step, save_path=plots_dir/f"{object_name}_pol_map.pdf")
        
        # Отрисовываем цветную карту со ВСЕМИ контурами, сколько бы их ни было
        self.plot_snr_colored_polarization_map(
            I_full, vx, vy, snr_pI_map, wcs, valid_contours, 
            object_name=object_name, step=step, 
            save_path=plots_dir/f"{object_name}_snr_colored_map.pdf"
        )

        # Выбираем глобальное число Маха для возврата (Combined или Core_1)
        global_target = "Combined_All_Cores" if "Combined_All_Cores" in sf_results_dict else "Core_1"
        ma_global = sf_results_dict.get(global_target, {}).get("MA_proxy") if global_target in sf_results_dict else None

        return psi_deg, sf_results_dict, {"Distance": distance_pc, "MA_proxy": ma_global}
    
    def process_all_files(self, distance_dict: dict, default_distance: float = 400.0, **kwargs):
        """
        Перебирает все .fits файлы в папке.
        distance_dict - словарь {'filename_prefix': distance_in_pc}
        """
        fits_files = list(self.base_dir.glob("*.fits")) + list(self.base_dir.glob("*.fit"))
        
        if not fits_files:
            print(f"No FITS files found in {self.base_dir}")
            return
            
        print(f"Found {len(fits_files)} files to process.")
        
        for file_path in fits_files:
            obj_name = file_path.stem
            # Ищем дистанцию в словаре, если нет - берем дефолтную
            dist = distance_dict.get(obj_name, default_distance)
            self.process_hawc_file(file_path, distance_pc=dist, **kwargs)

In [13]:
from pathlib import Path

# Укажите папку, где лежат ваши FITS файлы HAWC+
hawc_folder = "/Users/ildana/astro/nu/mach_number/data/FIELDMAPS_Data_Release/FIELDMAPS_Reprojection/Original"

# Инициализируем адаптированный класс
analyzer = HawcPolarizationAnalyzer(hawc_folder)

# Вы можете создать словарь дистанций для каждого филамента (файла)
distances = {
    "filament_1_file_name": 0,
    "filament_2_file_name": 0,
    "omc1_hawcp_bandE": 0
}

# 1. Либо запустить для одного конкретного файла:
file_to_process = Path(hawc_folder) / "G24_RA_Original.fits"
psi, sf_res, stats = analyzer.process_hawc_file(
    file_path=file_to_process, 
    distance_pc=0, 
    fwhm_fallback=14.1,       # Сработает, только если в FITS нет ключа BEAM
    pix_scale_fallback=4.55,  # Сработает, только если в FITS нет ключа PIXSCAL
    snr_threshold=3.0, 
    min_beams=10.0, 
    step=3
)

# 2. Либо запустить пакетную обработку всей папки сразу:
# analyzer.process_all_files(distance_dict=distances, default_distance=400.0)


PROCESSING: G24_RA_Original
-> Using Physics Scales: FWHM = 18.2", Pixel Scale = 4.55"/px
------> The countours searching is started
-> Area Thresholds: Physical (16.0 beams) = 290.0 px, Algorithmic = 144.0 px
-> Final Min Area Threshold set to: 290.0 pixels


In [44]:
from pathlib import Path

# Укажите папку, где лежат ваши FITS файлы HAWC+
hawc_folder = "/Users/ildana/astro/nu/mach_number/data/FIELDMAPS_Data_Release/FIELDMAPS_Reprojection/Original"

# Инициализируем адаптированный класс
analyzer = HawcPolarizationAnalyzer(hawc_folder)

# Вы можете создать словарь дистанций для каждого филамента (файла)
distances = {
    "filament_1_file_name": 0,
    "filament_2_file_name": 0,
    "omc1_hawcp_bandE": 0
}

# 1. Либо запустить для одного конкретного файла:
file_to_process = Path(hawc_folder) / "F0726_HA_POL_08018661_HAWEHWPE_MRG_154-237.fits"
psi, sf_res, stats = analyzer.process_hawc_file(
    file_path=file_to_process, 
    distance_pc=0, 
    fwhm_fallback=14.1,       # Сработает, только если в FITS нет ключа BEAM
    pix_scale_fallback=4.55,  # Сработает, только если в FITS нет ключа PIXSCAL
    snr_threshold=1.0, 
    min_beams=10.0, 
    step=3
)

# 2. Либо запустить пакетную обработку всей папки сразу:
# analyzer.process_all_files(distance_dict=distances, default_distance=400.0)


PROCESSING: F0726_HA_POL_08018661_HAWEHWPE_MRG_154-237
-> Using Physics Scales: FWHM = 18.2", Pixel Scale = 4.55"/px
------> The countours searching is started
-> Area Thresholds: Physical (16.0 beams) = 290.0 px, Algorithmic = 144.0 px
-> Final Min Area Threshold set to: 290.0 pixels


In [52]:
from pathlib import Path

# Укажите папку, где лежат ваши FITS файлы HAWC+
hawc_folder = "/Users/ildana/astro/nu/mach_number/data/FIELDMAPS_Data_Release/FIELDMAPS_Reprojection/Original"

# Инициализируем адаптированный класс
analyzer = HawcPolarizationAnalyzer(hawc_folder)

# 1. Либо запустить для одного конкретного файла:
file_to_process = Path(hawc_folder) / "Fil1_RA_Original.fits"

# Запуск автоматизированного конвейера
psi, sf_res, stats = analyzer.process_hawc_file(
    file_path=file_to_process, 
    distance_pc=0, 
    fwhm_fallback=14.1,       # Сработает, только если в FITS нет ключа BEAM
    pix_scale_fallback=4.55,  # Сработает, только если в FITS нет ключа PIXSCAL
    snr_threshold=3.0, 
    min_beams=10.0, 
    step=2

)

# 2. Либо запустить пакетную обработку всей папки сразу:
# analyzer.process_all_files(distance_dict=distances, default_distance=400.0)


PROCESSING: Fil1_RA_Original
-> Using Physics Scales: FWHM = 18.2", Pixel Scale = 4.55"/px
------> The countours searching is started
-> Area Thresholds: Physical (16.0 beams) = 290.0 px, Algorithmic = 144.0 px
-> Final Min Area Threshold set to: 290.0 pixels


## Fil1: structural functions with and without GETSOURCES sources

In [ ]:
import sys
from pathlib import Path

project_dir = Path("/Users/ildana/astro/nu/mach_number")
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

from getsources_sf_analysis import analyze_getsources_source_masks

# Берём 20 источников из финального reliable+tentative каталога GETSOURCES.
# Q и U не изменяются: footprints используются только как пространственная маска.
getsources_mask_results = analyze_getsources_source_masks(
    analyzer=analyzer,
    hawc_file=Path(hawc_folder) / "Fil1_RA_Original.fits",
    footprints_file=(
        project_dir / "getsources_runs/final/combo/sm.Fil1.sw.det.footprints.fits"
    ),
    source_catalog=(
        project_dir / "getsources_runs/final/combo/+catalogs/Fil1.sw.final.relitent.cat"
    ),
    distance_pc=0,
    fwhm_fallback=18.2,
    pix_scale_fallback=4.55,
    snr_threshold=3.0,
    min_beams=10.0,
    source_buffer_beams=0.5,
)

getsources_mask_results

In [15]:
from pathlib import Path

# Укажите папку, где лежат ваши FITS файлы HAWC+
hawc_folder = "/Users/ildana/astro/nu/mach_number/data/FIELDMAPS_Data_Release/FIELDMAPS_Reprojection/Original"

# Инициализируем адаптированный класс
analyzer = HawcPolarizationAnalyzer(hawc_folder)

# Вы можете создать словарь дистанций для каждого филамента (файла)
distances = {
    "filament_1_file_name": 0,
    "filament_2_file_name": 0,
    "omc1_hawcp_bandE": 0
}

# 1. Либо запустить для одного конкретного файла:
file_to_process = Path(hawc_folder) / "Fil10_RA_Original.fits"
psi, sf_res, stats = analyzer.process_hawc_file(
    file_path=file_to_process, 
    distance_pc=0, 
    fwhm_fallback=14.1,       # Сработает, только если в FITS нет ключа BEAM
    pix_scale_fallback=4.55,  # Сработает, только если в FITS нет ключа PIXSCAL
    snr_threshold=3.0, 
    min_beams=5.0, 
    step=3
)

# 2. Либо запустить пакетную обработку всей папки сразу:
# analyzer.process_all_files(distance_dict=distances, default_distance=400.0)


PROCESSING: Fil10_RA_Original
-> Using Physics Scales: FWHM = 18.2", Pixel Scale = 4.55"/px
------> The countours searching is started
-> Area Thresholds: Physical (16.0 beams) = 290.0 px, Algorithmic = 144.0 px
-> Final Min Area Threshold set to: 290.0 pixels


In [16]:
from pathlib import Path

# Укажите папку, где лежат ваши FITS файлы HAWC+
hawc_folder = "/Users/ildana/astro/nu/mach_number/data/FIELDMAPS_Data_Release/FIELDMAPS_Reprojection/Original"

# Инициализируем адаптированный класс
analyzer = HawcPolarizationAnalyzer(hawc_folder)


# 1. Либо запустить для одного конкретного файла:
file_to_process = Path(hawc_folder) / "Fil2_RA_Original.fits"
psi, sf_res, stats = analyzer.process_hawc_file(
    file_path=file_to_process, 
    distance_pc=0, 
    fwhm_fallback=14.1,       # Сработает, только если в FITS нет ключа BEAM
    pix_scale_fallback=4.55,  # Сработает, только если в FITS нет ключа PIXSCAL
    snr_threshold=3.0, 
    min_beams=5.0, 
    step=2
)

# 2. Либо запустить пакетную обработку всей папки сразу:
# analyzer.process_all_files(distance_dict=distances, default_distance=400.0)


PROCESSING: Fil2_RA_Original
-> Using Physics Scales: FWHM = 18.2", Pixel Scale = 4.55"/px
------> The countours searching is started
-> Area Thresholds: Physical (16.0 beams) = 290.0 px, Algorithmic = 144.0 px
-> Final Min Area Threshold set to: 290.0 pixels


In [17]:
from pathlib import Path

# Укажите папку, где лежат ваши FITS файлы HAWC+
hawc_folder = "/Users/ildana/astro/nu/mach_number/data/FIELDMAPS_Data_Release/FIELDMAPS_Reprojection/Original"

# Инициализируем адаптированный класс
analyzer = HawcPolarizationAnalyzer(hawc_folder)

# Вы можете создать словарь дистанций для каждого филамента (файла)
distances = {
    "filament_1_file_name": 0,
    "filament_2_file_name": 0,
    "omc1_hawcp_bandE": 0
}

# 1. Либо запустить для одного конкретного файла:
file_to_process = Path(hawc_folder) / "Fil4_RA_Original.fits"
psi, sf_res, stats = analyzer.process_hawc_file(
    file_path=file_to_process, 
    distance_pc=0, 
    fwhm_fallback=14.1,       # Сработает, только если в FITS нет ключа BEAM
    pix_scale_fallback=4.55,  # Сработает, только если в FITS нет ключа PIXSCAL
    snr_threshold=3.0, 
    min_beams=5.0, 
    step=3
)

# 2. Либо запустить пакетную обработку всей папки сразу:
# analyzer.process_all_files(distance_dict=distances, default_distance=400.0)


PROCESSING: Fil4_RA_Original
-> Using Physics Scales: FWHM = 18.2", Pixel Scale = 4.55"/px
------> The countours searching is started
-> Area Thresholds: Physical (16.0 beams) = 290.0 px, Algorithmic = 144.0 px
-> Final Min Area Threshold set to: 290.0 pixels


In [18]:
from pathlib import Path

# Укажите папку, где лежат ваши FITS файлы HAWC+
hawc_folder = "/Users/ildana/astro/nu/mach_number/data/FIELDMAPS_Data_Release/FIELDMAPS_Reprojection/Original"

# Инициализируем адаптированный класс
analyzer = HawcPolarizationAnalyzer(hawc_folder)

# Вы можете создать словарь дистанций для каждого филамента (файла)
distances = {
    "filament_1_file_name": 0,
    "filament_2_file_name": 0,
    "omc1_hawcp_bandE": 0
}

# 1. Либо запустить для одного конкретного файла:
file_to_process = Path(hawc_folder) / "Fil5_RA_Original.fits"
psi, sf_res, stats = analyzer.process_hawc_file(
    file_path=file_to_process, 
    distance_pc=0, 
    fwhm_fallback=14.1,       # Сработает, только если в FITS нет ключа BEAM
    pix_scale_fallback=4.55,  # Сработает, только если в FITS нет ключа PIXSCAL
    snr_threshold=3.0, 
    min_beams=5.0, 
    step=3
)

# 2. Либо запустить пакетную обработку всей папки сразу:
# analyzer.process_all_files(distance_dict=distances, default_distance=400.0)


PROCESSING: Fil5_RA_Original
-> Using Physics Scales: FWHM = 18.2", Pixel Scale = 4.55"/px
------> The countours searching is started
-> Area Thresholds: Physical (16.0 beams) = 290.0 px, Algorithmic = 144.0 px
-> Final Min Area Threshold set to: 290.0 pixels


In [19]:
from pathlib import Path

# Укажите папку, где лежат ваши FITS файлы HAWC+
hawc_folder = "/Users/ildana/astro/nu/mach_number/data/FIELDMAPS_Data_Release/FIELDMAPS_Reprojection/Original"

# Инициализируем адаптированный класс
analyzer = HawcPolarizationAnalyzer(hawc_folder)

# Вы можете создать словарь дистанций для каждого филамента (файла)
distances = {
    "filament_1_file_name": 0,
    "filament_2_file_name": 0,
    "omc1_hawcp_bandE": 0
}

# 1. Либо запустить для одного конкретного файла:
file_to_process = Path(hawc_folder) / "G47_RA_Original.fits"
psi, sf_res, stats = analyzer.process_hawc_file(
    file_path=file_to_process, 
    distance_pc=0, 
    fwhm_fallback=14.1,       # Сработает, только если в FITS нет ключа BEAM
    pix_scale_fallback=4.55,  # Сработает, только если в FITS нет ключа PIXSCAL
    snr_threshold=3.0, 
    min_beams=5.0, 
    step=3
)

# 2. Либо запустить пакетную обработку всей папки сразу:
# analyzer.process_all_files(distance_dict=distances, default_distance=400.0)


PROCESSING: G47_RA_Original
-> Using Physics Scales: FWHM = 18.2", Pixel Scale = 4.55"/px
------> The countours searching is started
-> Area Thresholds: Physical (16.0 beams) = 290.0 px, Algorithmic = 144.0 px
-> Final Min Area Threshold set to: 290.0 pixels


In [20]:
from pathlib import Path

# Укажите папку, где лежат ваши FITS файлы HAWC+
hawc_folder = "/Users/ildana/astro/nu/mach_number/data/FIELDMAPS_Data_Release/FIELDMAPS_Reprojection/Original"

# Инициализируем адаптированный класс
analyzer = HawcPolarizationAnalyzer(hawc_folder)

# Вы можете создать словарь дистанций для каждого филамента (файла)
distances = {
    "filament_1_file_name": 0,
    "filament_2_file_name": 0,
    "omc1_hawcp_bandE": 0
}

# 1. Либо запустить для одного конкретного файла:
file_to_process = Path(hawc_folder) / "G49_RA_Original.fits"
psi, sf_res, stats = analyzer.process_hawc_file(
    file_path=file_to_process, 
    distance_pc=0, 
    fwhm_fallback=14.1,       # Сработает, только если в FITS нет ключа BEAM
    pix_scale_fallback=4.55,  # Сработает, только если в FITS нет ключа PIXSCAL
    snr_threshold=3.0, 
    min_beams=5.0, 
    step=3
)

# 2. Либо запустить пакетную обработку всей папки сразу:
# analyzer.process_all_files(distance_dict=distances, default_distance=400.0)


PROCESSING: G49_RA_Original
-> Using Physics Scales: FWHM = 18.2", Pixel Scale = 4.55"/px
------> The countours searching is started
-> Area Thresholds: Physical (16.0 beams) = 290.0 px, Algorithmic = 144.0 px
-> Final Min Area Threshold set to: 290.0 pixels


In [21]:
from pathlib import Path

# Укажите папку, где лежат ваши FITS файлы HAWC+
hawc_folder = "/Users/ildana/astro/nu/mach_number/data/FIELDMAPS_Data_Release/FIELDMAPS_Reprojection/Original"

# Инициализируем адаптированный класс
analyzer = HawcPolarizationAnalyzer(hawc_folder)

# Вы можете создать словарь дистанций для каждого филамента (файла)
distances = {
    "filament_1_file_name": 0,
    "filament_2_file_name": 0,
    "omc1_hawcp_bandE": 0
}

# 1. Либо запустить для одного конкретного файла:
file_to_process = Path(hawc_folder) / "Fil8_RA_Original.fits"
psi, sf_res, stats = analyzer.process_hawc_file(
    file_path=file_to_process, 
    distance_pc=0, 
    fwhm_fallback=14.1,       # Сработает, только если в FITS нет ключа BEAM
    pix_scale_fallback=4.55,  # Сработает, только если в FITS нет ключа PIXSCAL
    snr_threshold=3.0, 
    min_beams=5.0, 
    step=3
)

# 2. Либо запустить пакетную обработку всей папки сразу:
# analyzer.process_all_files(distance_dict=distances, default_distance=400.0)


PROCESSING: Fil8_RA_Original
-> Using Physics Scales: FWHM = 18.2", Pixel Scale = 4.55"/px
------> The countours searching is started
-> Area Thresholds: Physical (16.0 beams) = 290.0 px, Algorithmic = 144.0 px
-> Final Min Area Threshold set to: 290.0 pixels


In [22]:
from pathlib import Path

# Укажите папку, где лежат ваши FITS файлы HAWC+
hawc_folder = "/Users/ildana/astro/nu/mach_number/data/FIELDMAPS_Data_Release/FIELDMAPS_Reprojection/Original"

# Инициализируем адаптированный класс
analyzer = HawcPolarizationAnalyzer(hawc_folder)

# Вы можете создать словарь дистанций для каждого филамента (файла)
distances = {
    "filament_1_file_name": 0,
    "filament_2_file_name": 0,
    "omc1_hawcp_bandE": 0
}

# 1. Либо запустить для одного конкретного файла:
file_to_process = Path(hawc_folder) / "Snake_RA_Original.fits"
psi, sf_res, stats = analyzer.process_hawc_file(
    file_path=file_to_process, 
    distance_pc=0, 
    fwhm_fallback=14.1,       # Сработает, только если в FITS нет ключа BEAM
    pix_scale_fallback=4.55,  # Сработает, только если в FITS нет ключа PIXSCAL
    snr_threshold=3.0, 
    min_beams=5.0, 
    step=3
)

# 2. Либо запустить пакетную обработку всей папки сразу:
# analyzer.process_all_files(distance_dict=distances, default_distance=400.0)


PROCESSING: Snake_RA_Original
-> Using Physics Scales: FWHM = 14.1", Pixel Scale = 4.55"/px
------> The countours searching is started
-> Area Thresholds: Physical (16.0 beams) = 174.1 px, Algorithmic = 144.0 px
-> Final Min Area Threshold set to: 174.1 pixels


In [23]:
import pandas as pd
from pathlib import Path

def build_latex_report_from_disk(base_dir: str = "/Users/ildana/astro/nu/mach_number/data/FIELDMAPS_Data_Release/FIELDMAPS_Reprojection/Original/hawc_report/plots", output_tex: str = "turbulence_report_sf.tex"):
    """
    Сканирует директории объектов, парсит CSV-таблицы (включая интенсивности, площади и пороги)
    и компилирует LaTeX-отчет с жестким форматированием и автоматическим переносом страниц.
    """
    root_path = Path(base_dir)
    if not root_path.exists():
        raise FileNotFoundError(f"Directory not found: {root_path}")

    # Преамбула с фиксацией плавающих объектов
    tex = [
        r"\documentclass[11pt,a4paper]{article}",
        r"\usepackage[utf8]{inputenc}",
        r"\usepackage[T2A]{fontenc}",
        r"\usepackage[english,russian]{babel}",
        r"\usepackage[margin=1.5cm]{geometry}",
        r"\usepackage{graphicx}",
        r"\usepackage{subcaption}",
        r"\usepackage{booktabs}",
        r"\usepackage{float}",
        r"\begin{document}",
        r"\title{Анализ магнитной гидродинамики (HAWC+)}",
        r"\author{Автоматизированный конвейер}",
        r"\date{\today}",
        r"\maketitle",
        r"\tableofcontents"
    ]

    object_dirs = sorted([d for d in root_path.iterdir() if d.is_dir()])

    for obj_dir in object_dirs:
        obj_name = obj_dir.name
        safe_name = obj_name.replace("_", r"\_")
        
        csv_files = list(obj_dir.glob(f"{obj_name}_*_analysis.csv"))
        if not csv_files:
            continue

        tex.append(r"\clearpage")
        tex.append(fr"\section{{{safe_name}}}")

        # Сбор данных по областям
        data = {}
        for csv in csv_files:
            reg_name = csv.stem.replace(f"{obj_name}_", "").replace("_analysis", "")
            df = pd.read_csv(csv)
            data[reg_name] = df.iloc[0].to_dict() if not df.empty else {}

        # Глобальные параметры
        global_key = "Combined_All_Cores" if "Combined_All_Cores" in data else list(data.keys())[0]
        dist = data[global_key].get("Distance", "N/A")
        
        ma_glob = data[global_key].get("MA_proxy", "N/A")
        if isinstance(ma_glob, float): ma_glob = f"{ma_glob:.3f}"
            
        # Извлечение вычисленного порога площади
        min_area_thresh = data[global_key].get("Min_Area_Threshold_px", "N/A")
        if isinstance(min_area_thresh, (float, int)) and pd.notna(min_area_thresh):
            min_area_thresh = f"{min_area_thresh:.0f}"

        # Таблица: Сводка параметров
        tex.append(r"\subsection*{Сводка параметров}")
        tex.append(r"\begin{table}[H]\centering\begin{tabular}{ll}\toprule")
        tex.append(r"\textbf{Метрика} & \textbf{Значение} \\\midrule")
        tex.append(fr"Дистанция (пк) & {dist} \\")
        tex.append(fr"Глобальное число Маха ($M_A$) & {ma_glob} \\")
        tex.append(fr"Адаптивный порог площади ($S_{{min}}$) & {min_area_thresh} px \\")
        tex.append(r"\bottomrule\end{tabular}\end{table}")

        # Таблица: Детализация по сегментам
        # Таблица: Детализация по сегментам
        tex.append(r"\subsection*{Детализация по областям}")
        tex.append(r"\begin{table}[H]\centering")
        tex.append(r"\resizebox{\textwidth}{!}{%") 
        # Увеличено количество колонок до 10
        tex.append(r"\begin{tabular}{l c c c c c c c c c}\toprule")
        tex.append(r"\textbf{Сегмент} & \textbf{$M_A$} & \textbf{$L$ (px)} & \textbf{$R_{\max}$ (px)} & \textbf{$l_A$ (px)} & \textbf{Площадь (px)} & \textbf{$I_{\max}$} & \textbf{$I_{\min}$} & \textbf{$I_{\rm med}$} & \textbf{Плато} \\\midrule")

        indiv = sorted([r for r in data.keys() if "Combined" not in r])
        comb = sorted([r for r in data.keys() if "Combined" in r])
        ordered = indiv + comb

        def fmt(val, fmt_str):
            """Безопасное форматирование значений."""
            return fmt_str.format(float(val)) if pd.notna(val) else "N/A"

        for reg in ordered:
            reg_safe = reg.replace("_", r"\_")
            row_data = data[reg]
            
            ma = row_data.get("MA_proxy", float('nan'))
            L = row_data.get("L_fit_px", float('nan'))
            rmax = row_data.get("R_max_pix", float('nan'))
            la = row_data.get("lA_fit_px", float('nan'))
            area = row_data.get("Area_pix", float('nan'))
            imax = row_data.get("I_max", float('nan'))
            imin = row_data.get("I_min", float('nan'))
            imed = row_data.get("I_median", float('nan'))
            
            # Строгая верификация наличия плато
            p_status = str(row_data.get("plateau_detected", ""))
            has_plateau = "Да" if p_status == "detected" or pd.notna(la) else "Нет"
            
            s_ma = fmt(ma, '{:.3f}')
            s_L = fmt(L, '{:.1f}')
            s_rmax = fmt(rmax, '{:.0f}')
            s_la = fmt(la, '{:.1f}')
            s_area = fmt(area, '{:.0f}')
            s_imax = fmt(imax, '{:.3e}')
            s_imin = fmt(imin, '{:.3e}')
            s_imed = fmt(imed, '{:.3e}')
            
            if "Combined" in reg:
                tex.append(r"\midrule")
                bold_cells = [fr"\textbf{{{c}}}" if c != "N/A" else "N/A" for c in [s_ma, s_L, s_rmax, s_la, s_area, s_imax, s_imin, s_imed]]
                bold_cells.append(fr"\textbf{{{has_plateau}}}")
                tex.append(fr"\textbf{{{reg_safe}}} & " + " & ".join(bold_cells) + r" \\")
            else:
                tex.append(fr"{reg_safe} & {s_ma} & {s_L} & {s_rmax} & {s_la} & {s_area} & {s_imax} & {s_imin} & {s_imed} & {has_plateau} \\")

        tex.append(r"\bottomrule\end{tabular}}")
        tex.append(r"\end{table}")

        # Карта сегментации
        map_path = obj_dir / f"{obj_name}_snr_colored_map.pdf"
        if map_path.exists():
            tex.append(r"\subsection*{Морфология}")
            tex.append(r"\begin{figure}[H]\centering")
            tex.append(fr"\includegraphics[width=0.85\textwidth]{{{map_path.as_posix()}}}")
            tex.append(r"\end{figure}")

        # Графики: SF, Npairs и SNR бок о бок (3 колонки)
        # Графики: SF, Npairs и SNR бок о бок (3 колонки)
        tex.append(r"\subsection*{Структурные функции и статистика}")
        
        # Флаг для проверки, были ли вообще добавлены графики в этом объекте
        plots_added = False 

        for reg in ordered:
            row_data = data.get(reg, {})
            
            # Строгий фильтр: отсекаем сегменты без обнаруженного плато.
            # Если lA_fit_px равен NaN и статус не "detected", графики не генерируем.
            la = row_data.get("lA_fit_px", float('nan'))
            plateau_status = str(row_data.get("plateau_detected", ""))
            
            if pd.isna(la) and plateau_status != "detected":
                continue
                
            plots_added = True
            reg_safe = reg.replace("_", r"\_")
            sf_file = obj_dir / f"{obj_name}_sf_{reg}.pdf"
            npairs_file = obj_dir / f"{obj_name}_npairs_drop_{reg}.pdf"
            snr_file = obj_dir / f"{obj_name}_snr_dist_{reg}.pdf"
            
            if sf_file.exists() and snr_file.exists() and npairs_file.exists():
                tex.append(r"\begin{figure}[H]\centering")
                
                tex.append(r"\begin{subfigure}{0.32\textwidth}")
                tex.append(fr"\includegraphics[width=\linewidth]{{{sf_file.as_posix()}}}")
                tex.append(r"\end{subfigure}\hfill")
                
                tex.append(r"\begin{subfigure}{0.32\textwidth}")
                tex.append(fr"\includegraphics[width=\linewidth]{{{npairs_file.as_posix()}}}")
                tex.append(r"\end{subfigure}\hfill")
                
                tex.append(r"\begin{subfigure}{0.32\textwidth}")
                tex.append(fr"\includegraphics[width=\linewidth]{{{snr_file.as_posix()}}}")
                tex.append(r"\end{subfigure}")
                
                tex.append(fr"\caption{{Сегмент \textbf{{{reg_safe}}}. Лево: SF. Центр: $N_{{pairs}}$. Право: SNR.}}")
                tex.append(r"\end{figure}")
                
            # Fallback для старых данных, где файла npairs может не быть
            elif sf_file.exists() and snr_file.exists():
                tex.append(r"\begin{figure}[H]\centering")
                tex.append(r"\begin{subfigure}{0.48\textwidth}")
                tex.append(fr"\includegraphics[width=\linewidth]{{{sf_file.as_posix()}}}")
                tex.append(r"\end{subfigure}\hfill")
                tex.append(r"\begin{subfigure}{0.48\textwidth}")
                tex.append(fr"\includegraphics[width=\linewidth]{{{snr_file.as_posix()}}}")
                tex.append(r"\end{subfigure}")
                tex.append(fr"\caption{{Сегмент \textbf{{{reg_safe}}}. Лево: SF. Право: SNR.}}")
                tex.append(r"\end{figure}")
        
        # Если ни одного плато не найдено, можно добавить пометку, чтобы подраздел не был пустым
        if not plots_added:
            tex.append(r"\vspace{1cm}")
            tex.append(r"\noindent\textit{Для данного объекта достоверных плато не обнаружено. Графики структурных функций опущены.}")

    tex.append(r"\end{document}")

    with open(output_tex, "w", encoding="utf-8") as f:
        f.write("\n".join(tex))

# Запуск
if __name__ == "__main__":
    build_latex_report_from_disk()

In [32]:
import pandas as pd
from pathlib import Path

def build_latex_report_from_disk(base_dir: str = "/Users/ildana/astro/nu/mach_number/data/FIELDMAPS_Data_Release/FIELDMAPS_Reprojection/Original/hawc_report/plots", output_tex: str = "turbulence_report.tex"):
    """
    Сканирует директории объектов, парсит CSV-таблицы (включая интенсивности, площади и пороги)
    и компилирует LaTeX-отчет с жестким форматированием и автоматическим переносом страниц.
    """
    root_path = Path(base_dir)
    if not root_path.exists():
        raise FileNotFoundError(f"Directory not found: {root_path}")

    # Преамбула с фиксацией плавающих объектов
    tex = [
        r"\documentclass[11pt,a4paper]{article}",
        r"\usepackage[utf8]{inputenc}",
        r"\usepackage[T2A]{fontenc}",
        r"\usepackage[english,russian]{babel}",
        r"\usepackage[margin=1.5cm]{geometry}",
        r"\usepackage{graphicx}",
        r"\usepackage{subcaption}",
        r"\usepackage{booktabs}",
        r"\usepackage{float}",
        r"\begin{document}",
        r"\title{Анализ магнитной гидродинамики (HAWC+)}",
        r"\author{Автоматизированный конвейер}",
        r"\date{\today}",
        r"\maketitle",
        r"\tableofcontents"
    ]

    object_dirs = sorted([d for d in root_path.iterdir() if d.is_dir()])

    for obj_dir in object_dirs:
        obj_name = obj_dir.name
        safe_name = obj_name.replace("_", r"\_")
        
        csv_files = list(obj_dir.glob(f"{obj_name}_*_analysis.csv"))
        if not csv_files:
            continue

        tex.append(r"\clearpage")
        tex.append(fr"\section{{{safe_name}}}")

        # Сбор данных по областям
        data = {}
        for csv in csv_files:
            reg_name = csv.stem.replace(f"{obj_name}_", "").replace("_analysis", "")
            df = pd.read_csv(csv)
            data[reg_name] = df.iloc[0].to_dict() if not df.empty else {}

        # Глобальные параметры
        global_key = "Combined_All_Cores" if "Combined_All_Cores" in data else list(data.keys())[0]
        dist = data[global_key].get("Distance", "N/A")
        
        ma_glob = data[global_key].get("MA_proxy", "N/A")
        if isinstance(ma_glob, float): ma_glob = f"{ma_glob:.3f}"
            
        # Извлечение вычисленного порога площади
        min_area_thresh = data[global_key].get("Min_Area_Threshold_px", "N/A")
        if isinstance(min_area_thresh, (float, int)) and pd.notna(min_area_thresh):
            min_area_thresh = f"{min_area_thresh:.0f}"

        # Таблица: Сводка параметров
        tex.append(r"\subsection*{Сводка параметров}")
        tex.append(r"\begin{table}[H]\centering\begin{tabular}{ll}\toprule")
        tex.append(r"\textbf{Метрика} & \textbf{Значение} \\\midrule")
        tex.append(fr"Дистанция (пк) & {dist} \\")
        tex.append(fr"Глобальное число Маха ($M_A$) & {ma_glob} \\")
        tex.append(fr"Адаптивный порог площади ($S_{{min}}$) & {min_area_thresh} px \\")
        tex.append(r"\bottomrule\end{tabular}\end{table}")

        # Таблица: Детализация по сегментам
        tex.append(r"\subsection*{Детализация по областям}")
        tex.append(r"\begin{table}[H]\centering")
        tex.append(r"\resizebox{\textwidth}{!}{%") 
        tex.append(r"\begin{tabular}{l c c c c c c c c}\toprule")
        tex.append(r"\textbf{Сегмент} & \textbf{$M_A$} & \textbf{$L$ (px)} & \textbf{$R_{\max}$ (px)} & \textbf{$l_A$ (px)} & \textbf{Площадь (px)} & \textbf{$I_{\max}$} & \textbf{$I_{\min}$} & \textbf{$I_{\rm med}$} \\\midrule")

        indiv = sorted([r for r in data.keys() if "Combined" not in r])
        comb = sorted([r for r in data.keys() if "Combined" in r])
        ordered = indiv + comb

        def fmt(val, fmt_str):
            """Безопасное форматирование значений."""
            return fmt_str.format(float(val)) if pd.notna(val) else "N/A"

        for reg in ordered:
            reg_safe = reg.replace("_", r"\_")
            row_data = data[reg]
            
            ma = row_data.get("MA_proxy", float('nan'))
            L = row_data.get("L_fit_px", float('nan'))
            rmax = row_data.get("R_max_pix", float('nan'))
            la = row_data.get("lA_fit_px", float('nan'))
            area = row_data.get("Area_pix", float('nan'))
            imax = row_data.get("I_max", float('nan'))
            imin = row_data.get("I_min", float('nan'))
            imed = row_data.get("I_median", float('nan'))
            
            s_ma = fmt(ma, '{:.3f}')
            s_L = fmt(L, '{:.1f}')
            s_rmax = fmt(rmax, '{:.0f}')
            s_la = fmt(la, '{:.1f}')
            s_area = fmt(area, '{:.0f}')
            s_imax = fmt(imax, '{:.3e}')
            s_imin = fmt(imin, '{:.3e}')
            s_imed = fmt(imed, '{:.3e}')
            
            if "Combined" in reg:
                tex.append(r"\midrule")
                bold_cells = [fr"\textbf{{{c}}}" if c != "N/A" else "N/A" for c in [s_ma, s_L, s_rmax, s_la, s_area, s_imax, s_imin, s_imed]]
                tex.append(fr"\textbf{{{reg_safe}}} & " + " & ".join(bold_cells) + r" \\")
            else:
                tex.append(fr"{reg_safe} & {s_ma} & {s_L} & {s_rmax} & {s_la} & {s_area} & {s_imax} & {s_imin} & {s_imed} \\")

        tex.append(r"\bottomrule\end{tabular}}")
        tex.append(r"\end{table}")

        # Карта сегментации
        map_path = obj_dir / f"{obj_name}_snr_colored_map.pdf"
        if map_path.exists():
            tex.append(r"\subsection*{Морфология}")
            tex.append(r"\begin{figure}[H]\centering")
            tex.append(fr"\includegraphics[width=0.85\textwidth]{{{map_path.as_posix()}}}")
            tex.append(r"\end{figure}")

        # Графики: SF, Npairs и SNR бок о бок (3 колонки)
        tex.append(r"\subsection*{Структурные функции и статистика}")
        for reg in ordered:
            reg_safe = reg.replace("_", r"\_")
            sf_file = obj_dir / f"{obj_name}_sf_{reg}.pdf"
            npairs_file = obj_dir / f"{obj_name}_npairs_drop_{reg}.pdf"
            snr_file = obj_dir / f"{obj_name}_snr_dist_{reg}.pdf"
            
            if sf_file.exists() and snr_file.exists() and npairs_file.exists():
                tex.append(r"\begin{figure}[H]\centering")
                
                tex.append(r"\begin{subfigure}{0.32\textwidth}")
                tex.append(fr"\includegraphics[width=\linewidth]{{{sf_file.as_posix()}}}")
                tex.append(r"\end{subfigure}\hfill")
                
                tex.append(r"\begin{subfigure}{0.32\textwidth}")
                tex.append(fr"\includegraphics[width=\linewidth]{{{npairs_file.as_posix()}}}")
                tex.append(r"\end{subfigure}\hfill")
                
                tex.append(r"\begin{subfigure}{0.32\textwidth}")
                tex.append(fr"\includegraphics[width=\linewidth]{{{snr_file.as_posix()}}}")
                tex.append(r"\end{subfigure}")
                
                tex.append(fr"\caption{{Сегмент \textbf{{{reg_safe}}}. Лево: SF. Центр: $N_{{pairs}}$. Право: SNR.}}")
                tex.append(r"\end{figure}")
                
            # Fallback для старых данных, где файла npairs может не быть
            elif sf_file.exists() and snr_file.exists():
                tex.append(r"\begin{figure}[H]\centering")
                tex.append(r"\begin{subfigure}{0.48\textwidth}")
                tex.append(fr"\includegraphics[width=\linewidth]{{{sf_file.as_posix()}}}")
                tex.append(r"\end{subfigure}\hfill")
                tex.append(r"\begin{subfigure}{0.48\textwidth}")
                tex.append(fr"\includegraphics[width=\linewidth]{{{snr_file.as_posix()}}}")
                tex.append(r"\end{subfigure}")
                tex.append(fr"\caption{{Сегмент \textbf{{{reg_safe}}}. Лево: SF. Право: SNR.}}")
                tex.append(r"\end{figure}")

    tex.append(r"\end{document}")

    with open(output_tex, "w", encoding="utf-8") as f:
        f.write("\n".join(tex))

# Запуск
if __name__ == "__main__":
    build_latex_report_from_disk()

In [ ]:
import pandas as pd
from pathlib import Path

def build_latex_report_from_disk(base_dir: str = "/Users/ildana/astro/nu/mach_number/data/FIELDMAPS_Data_Release/FIELDMAPS_Reprojection/Original/hawc_report/plots", output_tex: str = "turbulence_report.tex"):
    """
    Сканирует директории объектов, парсит CSV-таблицы (включая интенсивности и площади)
    и компилирует LaTeX-отчет с жестким форматированием и автоматическим переносом страниц.
    """
    root_path = Path(base_dir)
    if not root_path.exists():
        raise FileNotFoundError(f"Directory not found: {root_path}")

    # Преамбула с фиксацией плавающих объектов
    tex = [
        r"\documentclass[11pt,a4paper]{article}",
        r"\usepackage[utf8]{inputenc}",
        r"\usepackage[T2A]{fontenc}",
        r"\usepackage[english,russian]{babel}",
        r"\usepackage[margin=1.5cm]{geometry}",
        r"\usepackage{graphicx}",
        r"\usepackage{subcaption}",
        r"\usepackage{booktabs}",
        r"\usepackage{float}",
        r"\begin{document}",
        r"\title{Анализ магнитной гидродинамики (HAWC+)}",
        r"\author{Автоматизированный конвейер}",
        r"\date{\today}",
        r"\maketitle",
        r"\tableofcontents"
    ]

    object_dirs = sorted([d for d in root_path.iterdir() if d.is_dir()])

    for obj_dir in object_dirs:
        obj_name = obj_dir.name
        safe_name = obj_name.replace("_", r"\_")
        
        csv_files = list(obj_dir.glob(f"{obj_name}_*_analysis.csv"))
        if not csv_files:
            continue

        tex.append(r"\clearpage")
        tex.append(fr"\section{{{safe_name}}}")

        # Сбор данных по областям
        data = {}
        for csv in csv_files:
            reg_name = csv.stem.replace(f"{obj_name}_", "").replace("_analysis", "")
            df = pd.read_csv(csv)
            data[reg_name] = df.iloc[0].to_dict() if not df.empty else {}

        # Глобальные параметры
        global_key = "Combined_All_Cores" if "Combined_All_Cores" in data else list(data.keys())[0]
        dist = data[global_key].get("Distance", "N/A")
        ma_glob = data[global_key].get("MA_proxy", "N/A")
        if isinstance(ma_glob, float): ma_glob = f"{ma_glob:.3f}"

        tex.append(r"\subsection*{Сводка параметров}")
        tex.append(r"\begin{table}[H]\centering\begin{tabular}{ll}\toprule")
        tex.append(r"\textbf{Метрика} & \textbf{Значение} \\\midrule")
        tex.append(fr"Дистанция (пк) & {dist} \\")
        tex.append(fr"Глобальное число Маха ($M_A$) & {ma_glob} \\")
        tex.append(r"\bottomrule\end{tabular}\end{table}")

        # Таблица сегментов с новыми колонками и автомасштабированием
        tex.append(r"\subsection*{Детализация по областям}")
        tex.append(r"\begin{table}[H]\centering")
        tex.append(r"\resizebox{\textwidth}{!}{%") 
        tex.append(r"\begin{tabular}{l c c c c c c c c}\toprule") # Добавлена буква c
        tex.append(r"\textbf{Сегмент} & \textbf{$M_A$} & \textbf{$L$ (px)} & \textbf{$R_{\max}$ (px)} & \textbf{$l_A$ (px)} & \textbf{Площадь (px)} & \textbf{$I_{\max}$} & \textbf{$I_{\min}$} & \textbf{$I_{\rm med}$} \\\midrule")

        indiv = sorted([r for r in data.keys() if "Combined" not in r])
        comb = sorted([r for r in data.keys() if "Combined" in r])
        ordered = indiv + comb

        def fmt(val, fmt_str):
            """Безопасное форматирование значений."""
            return fmt_str.format(float(val)) if pd.notna(val) else "N/A"

        for reg in ordered:
            reg_safe = reg.replace("_", r"\_")
            row_data = data[reg]
            
            ma = row_data.get("MA_proxy", float('nan'))
            L = row_data.get("L_fit_px", float('nan'))
            rmax = row_data.get("R_max_pix", float('nan')) # <--- ДОБАВЛЕНО
            la = row_data.get("lA_fit_px", float('nan'))
            area = row_data.get("Area_pix", float('nan'))
            imax = row_data.get("I_max", float('nan'))
            imin = row_data.get("I_min", float('nan'))
            imed = row_data.get("I_median", float('nan'))
            
            s_ma = fmt(ma, '{:.3f}')
            s_L = fmt(L, '{:.1f}')
            s_rmax = fmt(rmax, '{:.0f}') # R_max — целое число
            s_la = fmt(la, '{:.1f}')
            s_area = fmt(area, '{:.0f}')
            s_imax = fmt(imax, '{:.3e}')
            s_imin = fmt(imin, '{:.3e}')
            s_imed = fmt(imed, '{:.3e}')
            
            if "Combined" in reg:
                tex.append(r"\midrule")
                bold_cells = [fr"\textbf{{{c}}}" if c != "N/A" else "N/A" for c in [s_ma, s_L, s_rmax, s_la, s_area, s_imax, s_imin, s_imed]]
                tex.append(fr"\textbf{{{reg_safe}}} & " + " & ".join(bold_cells) + r" \\")
            else:
                tex.append(fr"{reg_safe} & {s_ma} & {s_L} & {s_rmax} & {s_la} & {s_area} & {s_imax} & {s_imin} & {s_imed} \\")

        tex.append(r"\bottomrule\end{tabular}}")
        tex.append(r"\end{table}")

        # Карта сегментации
        map_path = obj_dir / f"{obj_name}_snr_colored_map.pdf"
        if map_path.exists():
            tex.append(r"\subsection*{Морфология}")
            tex.append(r"\begin{figure}[H]\centering")
            tex.append(fr"\includegraphics[width=0.85\textwidth]{{{map_path.as_posix()}}}")
            tex.append(r"\end{figure}")

        # Графики: SF и SNR бок о бок
        tex.append(r"\subsection*{Структурные функции}")
        for reg in ordered:
            reg_safe = reg.replace("_", r"\_")
            sf_file = obj_dir / f"{obj_name}_sf_{reg}.pdf"
            snr_file = obj_dir / f"{obj_name}_snr_dist_{reg}.pdf"
            
            if sf_file.exists() and snr_file.exists():
                tex.append(r"\begin{figure}[H]\centering")
                tex.append(r"\begin{subfigure}{0.48\textwidth}")
                tex.append(fr"\includegraphics[width=\linewidth]{{{sf_file.as_posix()}}}")
                tex.append(r"\end{subfigure}\hfill")
                tex.append(r"\begin{subfigure}{0.48\textwidth}")
                tex.append(fr"\includegraphics[width=\linewidth]{{{snr_file.as_posix()}}}")
                tex.append(r"\end{subfigure}")
                tex.append(fr"\caption{{Сегмент \textbf{{{reg_safe}}}. Лево: SF. Право: SNR.}}")
                tex.append(r"\end{figure}")

    tex.append(r"\end{document}")

    with open(output_tex, "w", encoding="utf-8") as f:
        f.write("\n".join(tex))

# Запуск
build_latex_report_from_disk()

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

# Настройка строгого научного стиля графиков
plt.rcParams.update({
    "text.usetex": False,  # <--- ИСПРАВЛЕНО: отключаем LaTeX-рендерер для текста
    "font.family": "serif",
    "axes.labelsize": 12,
    "axes.titlesize": 14,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "savefig.bbox": "tight"
})

def analyze_correlations(base_dir: str = "hawc_report/plots", output_dir: str = "hawc_report/analytics"):
    """
    Агрегирует данные по индивидуальным сегментам и строит диагностическую графику.
    """
    root_path = Path(base_dir)
    out_path = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    target_cols = [
        "MA_proxy", "L_fit_px", "R_max_pix", "lA_fit_px", 
        "Area_pix", "I_max", "I_min", "I_median"
    ]

    # 1. Сбор данных
    data_rows = []
    for csv_file in root_path.rglob("*_analysis.csv"):
        # Игнорируем комбинированные области, чтобы не загрязнять выборку дубликатами
        if "Combined" in csv_file.name:
            continue
            
        try:
            df = pd.read_csv(csv_file)
            if not df.empty:
                row = df.iloc[0].to_dict()
                row["Region_ID"] = csv_file.stem
                data_rows.append(row)
        except Exception as e:
            print(f"[ERROR] Чтение {csv_file.name}: {e}")

    if not data_rows:
        print("[!] Данные не найдены. Проверьте директорию.")
        return

    # Формируем DataFrame и очищаем от битых значений
    master_df = pd.DataFrame(data_rows)
    # Оставляем только целевые колонки (если они есть) + ID
    available_cols = [c for c in target_cols if c in master_df.columns]
    df_clean = master_df[available_cols].replace([np.inf, -np.inf], np.nan).dropna()

    print(f"[+] Собрано изолированных сегментов: {len(df_clean)}")

    # 2. Матрица корреляций (Heatmap)
    plt.figure(figsize=(10, 8))
    corr_matrix = df_clean[available_cols].corr(method='pearson')
    
    # Маска для скрытия верхней половины (избавление от визуального мусора)
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    
    sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", 
                vmin=-1, vmax=1, square=True, linewidths=.5, cbar_kws={"shrink": .8})
    plt.title("Матрица корреляций параметров турбулентности")
    plt.savefig(out_path / "01_correlation_heatmap.pdf")
    plt.close()

    # 3. Матрица рассеяния (Pairplot - стандарт МО)
    # Позволяет увидеть нелинейные зависимости и кластеры
    pp = sns.pairplot(df_clean[available_cols], diag_kind="kde", 
                      plot_kws={'alpha': 0.6, 's': 30, 'edgecolor': 'k'},
                      corner=True) # corner=True также убирает верхний треугольник
    pp.fig.suptitle("Матрица рассеяния (Pairplot)", y=1.02)
    plt.savefig(out_path / "02_parameter_pairplot.pdf")
    plt.close()

    # 4. Изолированный анализ Area_pix для поиска порога отсечения
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # А. Распределение площадей с логарифмической осью (поиск "хвоста" мелочи)
    sns.histplot(df_clean['Area_pix'], bins=30, kde=True, ax=axes[0], color='steelblue')
    axes[0].set_title("Распределение площадей сегментов")
    axes[0].set_xlabel("Area (px)")
    axes[0].set_ylabel("Частота")
    
    # Б. Зависимость Area от числа Маха (поиск шумового облака)
    # Если на малых площадях MA улетает в космос или падает в ноль - это шум.
    sns.scatterplot(data=df_clean, x='Area_pix', y='MA_proxy', 
                    alpha=0.7, edgecolor='k', ax=axes[1], color='crimson')
    axes[1].axhline(df_clean['MA_proxy'].median(), color='k', linestyle='--', alpha=0.5, label='Median MA')
    axes[1].set_title("Влияние размера области на оценку $M_A$")
    axes[1].set_xlabel("Area (px)")
    axes[1].set_ylabel("Alfvén Mach Number ($M_A$)")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(out_path / "03_area_cutoff_diagnostics.pdf")
    plt.close()
    
    print(f"[+] Аналитика сохранена в {out_path.resolve()}")

# Запуск
analyze_correlations(base_dir="/Users/ildana/astro/nu/mach_number/data/FIELDMAPS_Data_Release/FIELDMAPS_Reprojection/Original/hawc_report/plots")

In [ ]:
/Users/ildana/astro/nu/mach_number/hawc_report/analytics